In [1]:
import numpy as np
import xtrack as xt
import matplotlib.pyplot as plt
from xsuite_optics_imperfections import apply_errors, apply_orbit_correction, generate_monitor_misalignments
from xsuite_optics_imperfections import apply_monitor_misalignments, create_elements_switch, find_elements
import json

In [19]:
line_file = '100_lines_with_BPM_correctors/LEP3_Z_11f_wMarkers_wBPMs_wCorr.json'
line = xt.Line.from_json(line_file)
tt = line.get_table()
tw0 = line.twiss()

Loading line from dict:   0%|          | 0/14064 [00:00<?, ?it/s]

/home/tprebiba/miniforge3/envs/fcc-2025/lib/python3.12/site-packages/xtrack/beam_elements/elements.py:793: FutureWarning: `lag` (in degrees) is deprecated and will be removed in a future version. Please use `phase` (in radians) instead. If you see this warning while loading a saved line from a previous version of Xsuite, please regenerate the line with the current version to use phase instead of lag. Note that if both `lag` and `phase` are set, the effect is the sum of the two,  with `lag` converted to radians.  This deprecation is part of the interface cleanup in view of the 1.0 release.
  self.lag = lag


Done loading line from dict.           


In [20]:
# Hardcoded for now; to be improved later
markers_per_arc = [
    ('ir8_end', 'ir1_start'),
    ('ir1_end', 'ir2_start'),
    ('ir2_end', 'ir3_start'),
    ('ir3_end', 'ir4_start'),
    ('ir4_end', 'ir5_start'),
    ('ir5_end', 'ir6_start'),
    ('ir6_end', 'ir7_start'),
    ('ir7_end', 'ir8_start'),
]
markers_per_tr = [
    ('ir1_start', 'ir1_end'),
    ('ir2_start', 'ir2_end'),
    ('ir4_start', 'ir4_end'),
    ('ir5_start', 'ir5_end'),
    ('ir6_start', 'ir6_end'),
    ('ir8_start', 'ir8_end'),
]
markers_per_ff = [
    ('ir3_start', 'ir3_end'),
    ('ir7_start', 'ir7_end'),
]

all_arc_bends = find_elements(tt, marker_pairs = markers_per_arc)
all_arc_bends = all_arc_bends.rows[(all_arc_bends.element_type == 'Bend') | (all_arc_bends.element_type == 'RBend')]
all_arc_quads = find_elements(tt, marker_pairs = markers_per_arc, element_type = 'Quadrupole')
all_arc_sexts = find_elements(tt, marker_pairs = markers_per_arc, element_type = 'Sextupole')

all_tr_bends = find_elements(tt, marker_pairs = markers_per_tr, element_type='Bend')
#all_tr_bends = all_tr_bends.rows[(all_arc_bends.element_type == 'Bend') | (all_arc_bends.element_type == 'RBend')]
all_tr_quads = find_elements(tt, marker_pairs = markers_per_tr, element_type = 'Quadrupole')
all_tr_sexts = find_elements(tt, marker_pairs = markers_per_tr, element_type = 'Sextupole')

all_ff_bends = find_elements(tt, marker_pairs = markers_per_ff)
all_ff_bends = all_ff_bends.rows[(all_ff_bends.element_type == 'Bend') | (all_ff_bends.element_type == 'RBend')]
all_ff_quads_but_fd = find_elements(tt, marker_pairs = markers_per_ff, element_type = 'Quadrupole', except_pattern='q[f|d][0|1][a|b|c][r|l]')
all_fd_quads = find_elements(tt, pattern='q[f|d][0|1][a|b|c][r|l]',element_type = 'Quadrupole')
all_ff_sexts = find_elements(tt, marker_pairs = markers_per_ff, element_type = 'Sextupole')

print(f'Found in arcs: {len(all_arc_bends.name)} dipoles, {len(all_arc_quads.name)} quads, {len(all_arc_sexts.name)} sextupoles.')
print(f'Found in technical regions: {len(all_tr_bends.name)} dipoles, {len(all_tr_quads.name)} quads, {len(all_tr_sexts.name)} sextupoles.')
print(f'Found in final focusing: {len(all_ff_bends.name)} dipoles, {len(all_fd_quads.name)} FD quads and {len(all_ff_quads_but_fd.name)} other quads, {len(all_ff_sexts.name)} sextupoles.')

Found in arcs: 1680 dipoles, 1592 quads, 608 sextupoles.
Found in technical regions: 0 dipoles, 96 quads, 0 sextupoles.
Found in final focusing: 86 dipoles, 20 FD quads and 164 other quads, 88 sextupoles.


/eos/home-t/tprebiba/EPFL/14_LEP3/xsuite_optics_imperfections.py:97: UserWarning: Warning: No elements matched.
  warnings.warn("Warning: No elements matched.")


In [22]:
# Generate lines
#for seed in np.arange(1, 50, 1):
for seed in [4]:

    line_file = '100_lines_with_BPM_correctors/LEP3_Z_11f_wMarkers_wBPMs_wCorr.json'
    line = xt.Line.from_json(line_file)
    tt = line.get_table()
    tw0 = line.twiss()

    # ---------------------------------------------------------------------------- #
    #                    Apply misalignments and deactivate them                   #
    # ---------------------------------------------------------------------------- #
    # Arc dipoles
    all_arc_bends_names = apply_errors(line=line, pattern=None, seed=seed, element_names=all_arc_bends.name,
                                    sigmas=[1e-3, 1e-3, 0.5e-3, 1e-3], 
                                    attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                    switch_name='on_misalignment_dip_arc')
    line.vars['on_misalignment_dip_arc'] = 0
    # Arc quadrupoles
    all_arc_quads_names = apply_errors(line=line, pattern=None, seed=seed, element_names=all_arc_quads.name,
                                        sigmas=[50e-6, 50e-6, 100e-6, 50e-6], 
                                        attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                        switch_name='on_misalignment_quad_arc')
    line.vars['on_misalignment_quad_arc'] = 0
    # Arc sextupoles
    all_arc_sexts_names = apply_errors(line=line, pattern=None, seed=seed, element_names=all_arc_sexts.name,
                                        sigmas=[50e-6, 50e-6, 100e-6, 50e-6], 
                                        attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                        switch_name='on_misalignment_sext_arc')
    line.vars['on_misalignment_sext_arc'] = 0
    # TR quads
    all_tr_quads_names = apply_errors(line=line, pattern=None, seed=seed, element_names=all_tr_quads.name,
                                    sigmas=[100e-6, 100e-6, 100e-6, 100e-6], 
                                    attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                    switch_name='on_misalignment_quad_tr')
    line.vars['on_misalignment_quad_tr'] = 0
    # FF dipoles
    all_ff_bends_names = apply_errors(line=line, pattern=None, seed=seed, element_names=all_ff_bends.name,
                                    sigmas=[1e-3, 1e-3, 0.1e-3, 1e-3], 
                                    attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                    switch_name='on_misalignment_dip_ff',)
    line.vars['on_misalignment_dip_ff'] = 0
    # FD quads
    all_fd_quads_names = apply_errors(line=line, pattern=None, seed=seed, element_names=all_fd_quads.name,
                                    sigmas=[10e-6, 10e-6, 100e-6, 10e-6], 
                                    attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                    switch_name='on_misalignment_quad_fd')
    line.vars['on_misalignment_quad_fd'] = 0
    # FF quads
    all_ff_quads_but_fd_names = apply_errors(line=line, pattern=None, seed=seed, element_names=all_ff_quads_but_fd.name,
                                            sigmas=[30e-6, 30e-6, 100e-6, 30e-6], 
                                            attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                            switch_name='on_misalignment_quad_ff')
    line.vars['on_misalignment_quad_ff'] = 0
    # FF sextupoles
    all_ff_sexts_names = apply_errors(line=line, pattern=None, seed=seed, sigmas=[30e-6, 30e-6, 100e-6, 30e-6], element_names=all_ff_sexts.name,
                                    attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
                                    switch_name='on_misalignment_sext_ff')
    line.vars['on_misalignment_sext_ff'] = 0

    # ---------------------------------------------------------------------------- #
    #                     Apply field errors and deactivate them                   
    # ---------------------------------------------------------------------------- #
    # Arc dipoles
    _ = apply_errors(line=line, pattern=None, seed=seed, element_names=all_arc_bends.name,
                    sigmas=[1e-3], attrs=['k0'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_dip_arc')
    line.vars['on_field_error_dip_arc'] = 0
    # Arc quadrupoles
    _ = apply_errors(line=line, pattern=None, seed=seed, element_names=all_arc_quads.name,
                    sigmas=[2e-4], attrs=['k1'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_quad_arc')
    line.vars['on_field_error_quad_arc'] = 0
    # Arc sextupoles
    _ = apply_errors(line=line, pattern=None, seed=seed, element_names=all_arc_sexts.name,
                    sigmas=[2e-4], attrs=['k2'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_sext_arc')
    line.vars['on_field_error_sext_arc'] = 0
    # TR quads
    _ = apply_errors(line=line, pattern=None, seed=seed, 
                        sigmas=[2e-4], attrs=['k1'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                        apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                        switch_name='on_field_error_quad_tr',
                        element_names=all_tr_quads.name)
    line.vars['on_field_error_quad_tr'] = 0
    # FF dipoles
    _ = apply_errors(line=line, pattern=None, seed=seed, 
                        sigmas=[1e-3], attrs=['k0'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                        apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                        switch_name='on_field_error_dip_ff',
                        element_names=all_ff_bends.name)
    line.vars['on_field_error_dip_ff'] = 0
    # FD quads
    _ = apply_errors(line=line, pattern=None, seed=seed, 
                        sigmas=[0.1e-4], attrs=['k1'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                        apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                        switch_name='on_field_error_quad_fd',
                        element_names=all_fd_quads.name)
    line.vars['on_field_error_quad_fd'] = 0
    # FF quads
    _ = apply_errors(line=line, pattern=None, seed=seed, 
                        sigmas=[1e-4], attrs=['k1'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                        apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                        switch_name='on_field_error_quad_ff',
                        element_names=all_ff_quads_but_fd.name)
    line.vars['on_field_error_quad_ff'] = 0
    # FF sextupoles
    _ = apply_errors(line=line, pattern=None, seed=seed, 
                        sigmas=[1e-4], attrs=['k2'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                        apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                        switch_name='on_field_error_sext_ff',
                        element_names=all_ff_sexts.name)
    line.vars['on_field_error_sext_ff'] = 0


    # Create sextupole switches to easily deactivate them
    create_elements_switch(line=line, line_table=tt, switch_name='on_arc_sextupoles', marker_pairs=markers_per_arc, element_type='Sextupole')
    create_elements_switch(line=line, line_table=tt, switch_name='on_tr_ff_sextupoles', marker_pairs=markers_per_tr+markers_per_ff, element_type='Sextupole')
    create_elements_switch(line=line, line_table=tt, switch_name='on_cr_sextupoles', pattern='scr', element_type='Sextupole') # only crab sextupoles
    # Deactivate Sextupoles
    line.vars['on_arc_sextupoles'] = 0
    line.vars['on_tr_ff_sextupoles'] = 0
    line.vars['on_cr_sextupoles'] = 0

    # Set chromatic properties to False
    line.twiss_default['compute_chromatic_properties'] = False
    print('Switched to compute_chromatic_properties=False.')
    line.twiss_default['method'] = '4d'
    print('Switched to method=4d.')
    tw0 = line.twiss()

    line.vars['on_misalignment_dip_arc'] = 1
    line.vars['on_misalignment_quad_arc'] = 1
    line.vars['on_misalignment_sext_arc'] = 1
    line.vars['on_misalignment_quad_tr'] = 1
    line.vars['on_misalignment_dip_ff'] = 1
    line.vars['on_misalignment_quad_fd'] = 1
    line.vars['on_misalignment_quad_ff'] = 1
    line.vars['on_misalignment_sext_ff'] = 1

    line.vars['on_field_error_dip_arc'] = 1
    line.vars['on_field_error_quad_arc'] = 1
    line.vars['on_field_error_sext_arc'] = 1
    line.vars['on_field_error_quad_tr'] = 1
    line.vars['on_field_error_dip_ff'] = 1
    line.vars['on_field_error_quad_fd'] = 1
    line.vars['on_field_error_quad_ff'] = 1
    line.vars['on_field_error_sext_ff'] = 1

    # Add BPM misalignment (inheriting from quads + additional)
    monitor_alignment = generate_monitor_misalignments(line, pattern='bpm', attrs=['shift_x', 'shift_y', 'rot_s_rad'], 
                                                        line_table=tt, element_type='Marker')
    monitor_alignment = apply_monitor_misalignments(monitor_alignment, seed, sigmas=[10e-6, 10e-6, 10e-6])

    # Check perturbed lattice
    # tw = line.twiss()
    # f, axs = plt.subplots(2,1, figsize=(8, 5), sharex=True)
    # fontsize=12
    # ax = axs[0]
    # ax.set_ylabel('x (mm)', fontsize=fontsize)
    # ax.tick_params(axis='both', which='major', labelsize=fontsize)
    # ax.plot(tw0.s, tw0.x*1e3)
    # ax.plot(tw.s, tw.x*1e3)
    # ax = axs[1]
    # ax.set_xlabel('s (m)', fontsize=fontsize)
    # ax.set_ylabel('y (mm)', fontsize=fontsize)
    # ax.tick_params(axis='both', which='major', labelsize=fontsize)
    # ax.plot(tw0.s, tw0.y*1e3)
    # ax.plot(tw.s, tw.y*1e3)
    # f.tight_layout()

    line.to_json('101_lines_with_imperfections_deactivated_sexts/LEP3_Z_11f_seed%s_deactSext.json'%int(seed))
    # save monitor_aligntment to .json
    with open('101_lines_with_imperfections_deactivated_sexts/BPM_alignment_errors_seed%s.json'%int(seed), 'w') as fid:
        json.dump(monitor_alignment, fid)

Loading line from dict:   0%|          | 0/14064 [00:00<?, ?it/s]

Done loading line from dict.           
608 elements found for switch on_arc_sextupoles
88 elements found for switch on_tr_ff_sextupoles
16 elements found for switch on_cr_sextupoles
Switched to compute_chromatic_properties=False.
Switched to method=4d.
